# Divergencia KL y entropía cruzada

Este cuaderno compara distribuciones discretas pequeñas. Calcularemos entropía, entropía cruzada y divergencia KL.

La interpretación práctica será: cuántos bits extra pagamos cuando usamos un modelo `Q` para datos generados por `P`.

In [ ]:
import math


def validar(distribucion):
    total = sum(distribucion.values())
    if not math.isclose(total, 1.0, abs_tol=1e-9):
        raise ValueError(f"La distribución debe sumar 1; suma {total}.")
    if any(p < 0 for p in distribucion.values()):
        raise ValueError("Las probabilidades no pueden ser negativas.")


def entropia(p):
    validar(p)
    return -sum(prob * math.log2(prob) for prob in p.values() if prob > 0)


def entropia_cruzada(p, q):
    validar(p)
    validar(q)
    total = 0
    for simbolo, prob_p in p.items():
        prob_q = q.get(simbolo, 0)
        if prob_p > 0 and prob_q == 0:
            return math.inf
        if prob_p > 0:
            total -= prob_p * math.log2(prob_q)
    return total


def divergencia_kl(p, q):
    h_pq = entropia_cruzada(p, q)
    return h_pq - entropia(p) if math.isfinite(h_pq) else math.inf

## Modelos para una moneda

La distribución real `P` es una moneda equilibrada. Comparamos varios modelos `Q`.

In [ ]:
p_real = {"cara": 0.5, "cruz": 0.5}
modelos = {
    "correcto": {"cara": 0.5, "cruz": 0.5},
    "sesgo leve": {"cara": 0.6, "cruz": 0.4},
    "sesgo fuerte": {"cara": 0.9, "cruz": 0.1},
    "invertido fuerte": {"cara": 0.1, "cruz": 0.9},
}

print(f"H(P) = {entropia(p_real):.4f} bits")
for nombre, q in modelos.items():
    h_pq = entropia_cruzada(p_real, q)
    kl = divergencia_kl(p_real, q)
    print(f"{nombre:16s} H(P,Q) = {h_pq:.4f}  D_KL(P||Q) = {kl:.4f}")

## La dirección importa

`D_KL(P || Q)` y `D_KL(Q || P)` normalmente no son iguales.

In [ ]:
p = {"A": 0.8, "B": 0.2}
q = {"A": 0.5, "B": 0.5}

print("D_KL(P || Q):", round(divergencia_kl(p, q), 4))
print("D_KL(Q || P):", round(divergencia_kl(q, p), 4))

## Probabilidad cero

Si el modelo declara imposible algo que la fuente real sí puede producir, la divergencia se vuelve infinita.

In [ ]:
p = {"A": 0.9, "B": 0.1}
q = {"A": 1.0, "B": 0.0}

print("H(P,Q):", entropia_cruzada(p, q))
print("D_KL(P||Q):", divergencia_kl(p, q))

In [ ]:
probabilidades = [i / 100 for i in range(1, 100)]
kl_valores = [divergencia_kl(p_real, {"cara": r, "cruz": 1 - r}) for r in probabilidades]

try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib no está instalado; se omite la gráfica.")
else:
    plt.figure(figsize=(7, 4))
    plt.plot(probabilidades, kl_valores)
    plt.xlabel("Q(cara)")
    plt.ylabel("D_KL(P||Q)")
    plt.title("Coste extra al modelar una moneda equilibrada")
    plt.grid(True, alpha=0.3)
    plt.show()

## Para experimentar

1. Cambia `p_real` por una moneda sesgada.
2. Añade modelos que se aproximen gradualmente a `P`.
3. Comprueba dónde se alcanza el mínimo de `D_KL(P || Q)`.